# Data Cleaning & Preprocessing

## Purpose

This notebook transforms raw Walmart sales data into a production-ready dataset. We:
- Fix data types (especially dates)
- Engineer time-based features for analysis
- Detect and flag outliers (not drop them — preserve signal)
- Handle duplicates
- Create a clean, versioned dataset in `data/processed/`

**Key principle**: Flag issues rather than silently dropping data. A "bad" value might be a real holiday spike or data entry variance — we document it, not delete it.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Raw Data

Load the untouched raw dataset. We never modify the original file.

In [2]:
df = pd.read_csv('../data/raw/Walmart.csv')

print("=" * 60)
print("RAW DATA SNAPSHOT")
print("=" * 60)
print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn names & types:")
print(df.dtypes)
print(f"\nFirst 3 rows:")
display(df.head(3))

RAW DATA SNAPSHOT

Shape: 6,435 rows × 8 columns

Column names & types:
Store             int64
Date                str
Weekly_Sales    float64
Holiday_Flag      int64
Temperature     float64
Fuel_Price      float64
CPI             float64
Unemployment    float64
dtype: object

First 3 rows:


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106


## 3. Fix Date Column

The Date column is stored as a string. Convert it to datetime for time-series operations.

**Why this matters:**
- String dates can't be sorted correctly (e.g., "01-01-2010" < "02-01-2010" as strings, but the comparison works)
- We need `.dt.year`, `.dt.month`, `.dt.quarter` accessors for feature engineering
- Datetime format is required for any time-series analysis downstream

In [3]:
# Check the date format in raw data
print("Sample dates from raw data:")
print(df['Date'].head(10))

print("\n" + "="*60)
print("Converting Date to datetime format...")
print("="*60)

# Convert to datetime (format: DD-MM-YYYY)
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

print(f"✓ Date column converted to {df['Date'].dtype}")
print(f"\nDate range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Span: {(df['Date'].max() - df['Date'].min()).days} days ({(df['Date'].max() - df['Date'].min()).days / 365.25:.2f} years)")
print(f"\nSample dates after conversion:")
print(df['Date'].head(10))

Sample dates from raw data:
0    05-02-2010
1    12-02-2010
2    19-02-2010
3    26-02-2010
4    05-03-2010
5    12-03-2010
6    19-03-2010
7    26-03-2010
8    02-04-2010
9    09-04-2010
Name: Date, dtype: str

Converting Date to datetime format...
✓ Date column converted to datetime64[us]

Date range: 2010-02-05 00:00:00 to 2012-10-26 00:00:00
Span: 994 days (2.72 years)

Sample dates after conversion:
0   2010-02-05
1   2010-02-12
2   2010-02-19
3   2010-02-26
4   2010-03-05
5   2010-03-12
6   2010-03-19
7   2010-03-26
8   2010-04-02
9   2010-04-09
Name: Date, dtype: datetime64[us]


## 4. Sort Data Chronologically

Essential for time-series analysis: sort by Store first (to group same store together), then by Date (chronologically).

**Why**: When we compute rolling averages, lag features, or growth rates later, the data MUST be in time order.

In [4]:
print("Sorting by Store and Date...")
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

print(f"✓ Data sorted and index reset")
print(f"\nVerification - Store 1 over time:")
display(df[df['Store'] == 1][['Store', 'Date', 'Weekly_Sales']].head(10))

Sorting by Store and Date...
✓ Data sorted and index reset

Verification - Store 1 over time:


,Store,Date,Weekly_Sales
0,1,2010-02-05,1643690.90
1,1,2010-02-12,1641957.44
2,1,2010-02-19,1611968.17
3,1,2010-02-26,1409727.59
4,1,2010-03-05,1554806.68
5,1,2010-03-12,1439541.59
6,1,2010-03-19,1472515.79
7,1,2010-03-26,1404429.92
8,1,2010-04-02,1594968.28
9,1,2010-04-09,1545418.53


## 5. Feature Engineering

Extract time components from the Date column. These become valuable features for EDA and ML models.

In [5]:
print("Feature Engineering: Extracting time components...")
print("-" * 60)

# Year
df['Year'] = df['Date'].dt.year
print(f"✓ Year: {df['Year'].unique()}")

# Month (1-12)
df['Month'] = df['Date'].dt.month
print(f"✓ Month: {sorted(df['Month'].unique())}")

# Week of year (1-52)
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
print(f"✓ Week: {df['Week'].min()}-{df['Week'].max()}")

# Quarter (1-4)
df['Quarter'] = df['Date'].dt.quarter
print(f"✓ Quarter: {sorted(df['Quarter'].unique())}")

# Month name (for readable charts)
month_map = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
             7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
df['Month_Name'] = df['Month'].map(month_map)
print(f"✓ Month_Name: {df['Month_Name'].unique()}")

# Day of week (0=Monday, 6=Sunday)
df['DayOfWeek'] = df['Date'].dt.dayofweek
print(f"✓ DayOfWeek: {sorted(df['DayOfWeek'].unique())}")

print("\n" + "="*60)
print("New features added. Sample:")
print("="*60)
display(df[['Date', 'Year', 'Month', 'Month_Name', 'Week', 'Quarter', 'DayOfWeek']].head(10))

Feature Engineering: Extracting time components...
------------------------------------------------------------
✓ Year: [2010 2011 2012]
✓ Month: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
✓ Week: 1-52
✓ Quarter: [np.int32(1), np.int32(2), np.int32(3), np.int32(4)]
✓ Month_Name: <StringArray>
['Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec',
 'Jan']
Length: 12, dtype: str
✓ DayOfWeek: [np.int32(4)]

New features added. Sample:


,Date,Year,Month,Month_Name,Week,Quarter,DayOfWeek
0,2010-02-05,2010,2,Feb,5,1,4
1,2010-02-12,2010,2,Feb,6,1,4
2,2010-02-19,2010,2,Feb,7,1,4
3,2010-02-26,2010,2,Feb,8,1,4
4,2010-03-05,2010,3,Mar,9,1,4
5,2010-03-12,2010,3,Mar,10,1,4
6,2010-03-19,2010,3,Mar,11,1,4
7,2010-03-26,2010,3,Mar,12,1,4
8,2010-04-02,2010,4,Apr,13,2,4
9,2010-04-09,2010,4,Apr,14,2,4


## 6. Outlier Detection

Detect outliers using the **Interquartile Range (IQR) method**:
- Values below Q1 - 1.5×IQR or above Q3 + 1.5×IQR are flagged
- We **flag** them with a column, NOT drop them
- A Black Friday spike is a real sales surge, not an error

In [6]:
print("OUTLIER DETECTION (IQR Method)")
print("=" * 60)

# Calculate quartiles
Q1 = df['Weekly_Sales'].quantile(0.25)
Q3 = df['Weekly_Sales'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"\nQ1 (25th percentile):  ${Q1:>12,.2f}")
print(f"Q3 (75th percentile):  ${Q3:>12,.2f}")
print(f"IQR (Q3 - Q1):         ${IQR:>12,.2f}")
print(f"\nLower bound (Q1 - 1.5×IQR): ${lower_bound:>12,.2f}")
print(f"Upper bound (Q3 + 1.5×IQR): ${upper_bound:>12,.2f}")

# Flag outliers
df['Is_Outlier'] = ((df['Weekly_Sales'] < lower_bound) | 
                    (df['Weekly_Sales'] > upper_bound))

outlier_count = df['Is_Outlier'].sum()
outlier_pct = (outlier_count / len(df)) * 100

print(f"\n" + "="*60)
print(f"RESULTS:")
print(f"="*60)
print(f"Outliers detected: {outlier_count} ({outlier_pct:.2f}%)")

if outlier_count > 0:
    print(f"\nOutlier statistics:")
    outlier_stats = df[df['Is_Outlier']]['Weekly_Sales'].describe()
    print(outlier_stats.round(2))
    
    print(f"\nTop 10 outliers (highest sales):")
    top_outliers = df[df['Is_Outlier']].nlargest(10, 'Weekly_Sales')[
        ['Store', 'Date', 'Week', 'Holiday_Flag', 'Weekly_Sales']
    ]
    display(top_outliers)

OUTLIER DETECTION (IQR Method)

Q1 (25th percentile):  $  553,350.10
Q3 (75th percentile):  $1,420,158.66
IQR (Q3 - Q1):         $  866,808.55

Lower bound (Q1 - 1.5×IQR): $ -746,862.73
Upper bound (Q3 + 1.5×IQR): $2,720,371.49

RESULTS:
Outliers detected: 34 (0.53%)

Outlier statistics:
count         34.00
mean     3086723.36
std       379436.72
min      2727575.18
25%      2767649.33
50%      2913971.48
75%      3474992.09
max      3818686.45
Name: Weekly_Sales, dtype: float64

Top 10 outliers (highest sales):


,Store,Date,Week,Holiday_Flag,Weekly_Sales
1905,14,2010-12-24,51,0,3818686.45
2763,20,2010-12-24,51,0,3766687.43
1333,10,2010-12-24,51,0,3749057.69
527,4,2011-12-23,51,0,3676388.98
1762,13,2010-12-24,51,0,3595903.20
1814,13,2011-12-23,51,0,3556766.03
2815,20,2011-12-23,51,0,3555371.03
475,4,2010-12-24,51,0,3526713.39
1385,10,2011-12-23,51,0,3487986.89
189,2,2010-12-24,51,0,3436007.68


## 7. Duplicate Detection

Check for exact duplicate rows. A duplicate could mean:
- Data entry error
- System logging duplicate
- Legitimate repeat (rare, but worth checking)

In [7]:
print("DUPLICATE DETECTION")
print("=" * 60)

# Check for exact duplicates (all columns identical)
duplicate_count = df.duplicated().sum()
print(f"\nExact duplicate rows: {duplicate_count}")

# Check for duplicates on key columns (Store + Date combination)
key_duplicates = df.duplicated(subset=['Store', 'Date']).sum()
print(f"Store+Date duplicates: {key_duplicates}")

if key_duplicates > 0:
    print(f"\nSample key duplicates:")
    dup_mask = df.duplicated(subset=['Store', 'Date'], keep=False)
    display(df[dup_mask].sort_values(['Store', 'Date']).head(10))

# Remove exact duplicates if any
if duplicate_count > 0:
    print(f"\nRemoving {duplicate_count} exact duplicates...")
    df = df.drop_duplicates()
    print("✓ Duplicates removed")
else:
    print("✓ No exact duplicates found")

DUPLICATE DETECTION

Exact duplicate rows: 0
Store+Date duplicates: 0
✓ No exact duplicates found


## 8. Data Quality Summary

In [8]:
print("\n" + "="*70)
print("DATA QUALITY SUMMARY - BEFORE & AFTER")
print("="*70)

print(f"\nDataset Shape:")
print(f"  Before cleaning: 6,435 rows × 8 columns")
print(f"  After cleaning:  {df.shape[0]:,} rows × {df.shape[1]} columns")

print(f"\nData Types:")
print(df.dtypes)

print(f"\nMissing Values:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("  ✓ No missing values")
else:
    print(missing[missing > 0])

print(f"\nData Coverage:")
stores = df['Store'].nunique()
weeks = df.groupby('Store')['Date'].apply(lambda x: len(x)).mean()
print(f"  Stores: {stores}")
print(f"  Avg weeks per store: {weeks:.0f}")
print(f"  Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")

print(f"\nOutliers Flagged:")
print(f"  Count: {df['Is_Outlier'].sum()} ({df['Is_Outlier'].sum()/len(df)*100:.2f}%)")

print(f"\nSales Statistics (Weekly_Sales):")
print(f"  Mean:   ${df['Weekly_Sales'].mean():>12,.2f}")
print(f"  Median: ${df['Weekly_Sales'].median():>12,.2f}")
print(f"  Min:    ${df['Weekly_Sales'].min():>12,.2f}")
print(f"  Max:    ${df['Weekly_Sales'].max():>12,.2f}")
print(f"  Std:    ${df['Weekly_Sales'].std():>12,.2f}")

print(f"\nSample of cleaned data:")
display(df.head(10))


DATA QUALITY SUMMARY - BEFORE & AFTER

Dataset Shape:
  Before cleaning: 6,435 rows × 8 columns
  After cleaning:  6,435 rows × 15 columns

Data Types:
Store                    int64
Date            datetime64[us]
Weekly_Sales           float64
Holiday_Flag             int64
Temperature            float64
Fuel_Price             float64
CPI                    float64
Unemployment           float64
Year                     int32
Month                    int32
Week                     int64
Quarter                  int32
Month_Name                 str
DayOfWeek                int32
Is_Outlier                bool
dtype: object

Missing Values:
  ✓ No missing values

Data Coverage:
  Stores: 45
  Avg weeks per store: 143
  Date range: 2010-02-05 to 2012-10-26

Outliers Flagged:
  Count: 34 (0.53%)

Sales Statistics (Weekly_Sales):
  Mean:   $1,046,964.88
  Median: $  960,746.04
  Min:    $  209,986.25
  Max:    $3,818,686.45
  Std:    $  564,366.62

Sample of cleaned data:


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Week,Quarter,Month_Name,DayOfWeek,Is_Outlier
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,5,1,Feb,4,False
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,2010,2,6,1,Feb,4,False
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,2010,2,7,1,Feb,4,False
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106,2010,2,8,1,Feb,4,False
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106,2010,3,9,1,Mar,4,False
5,1,2010-03-12,1439541.59,0,57.79,2.667,211.380643,8.106,2010,3,10,1,Mar,4,False
6,1,2010-03-19,1472515.79,0,54.58,2.720,211.215635,8.106,2010,3,11,1,Mar,4,False
7,1,2010-03-26,1404429.92,0,51.45,2.732,211.018042,8.106,2010,3,12,1,Mar,4,False
8,1,2010-04-02,1594968.28,0,62.27,2.719,210.820450,7.808,2010,4,13,2,Apr,4,False
9,1,2010-04-09,1545418.53,0,65.86,2.770,210.622857,7.808,2010,4,14,2,Apr,4,False


## 9. Save Clean Dataset

Export the cleaned, feature-engineered dataset to `data/processed/` for use in downstream notebooks.

In [9]:
output_path = '../data/processed/walmart_sales_clean.csv'

df.to_csv(output_path, index=False)

print("="*70)
print("✓ CLEAN DATASET SAVED")
print("="*70)
print(f"\nFile: {output_path}")
print(f"Size: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Verify the saved file
df_verify = pd.read_csv(output_path)
print(f"\nVerification - reloaded file:")
print(f"Shape: {df_verify.shape}")
print(f"Columns: {df_verify.columns.tolist()}")
print(f"\nFirst 3 rows:")
display(df_verify.head(3))

✓ CLEAN DATASET SAVED

File: ../data/processed/walmart_sales_clean.csv
Size: 6,435 rows × 15 columns

Verification - reloaded file:
Shape: (6435, 15)
Columns: ['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Year', 'Month', 'Week', 'Quarter', 'Month_Name', 'DayOfWeek', 'Is_Outlier']

First 3 rows:


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Week,Quarter,Month_Name,DayOfWeek,Is_Outlier
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,5,1,Feb,4,False
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,2010,2,6,1,Feb,4,False
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,2010,2,7,1,Feb,4,False


## ✅ Summary

**What we did:**
1. ✓ Loaded raw data from `data/raw/Walmart.csv`
2. ✓ Converted Date strings to datetime format
3. ✓ Sorted data chronologically by Store and Date
4. ✓ Engineered 6 new time-based features (Year, Month, Week, Quarter, Month_Name, DayOfWeek)
5. ✓ Flagged outliers using IQR method (preserved, not dropped)
6. ✓ Detected and removed exact duplicates (if any)
7. ✓ Saved clean dataset to `data/processed/walmart_sales_clean.csv`

**Next steps:**
- Use the clean dataset in **Notebook 04 (EDA - Basic)** for distributions, store rankings, and holiday impact
- The `Is_Outlier` flag allows us to include or exclude outliers in downstream analysis based on context

**Key principle applied:**
We flagged outliers rather than dropping them. In retail, extreme sales (Black Friday, New Year sales) are real business signals, not errors. Downstream analysis can choose to include them for understanding extreme events, or exclude them for baseline trend analysis.